# 03 – Hyperparameter Tuning

## Purpose

This module takes the **top-N baseline models** produced by `src/training.py`
and searches for better hyperparameters using `RandomizedSearchCV`.
It returns:

- `comparison_df` – a tidy DataFrame comparing before/after RMSE, MAE and R²
- `tuned_models` – a dict of `{model_name: best_estimator_}` ready for
  downstream evaluation and persistence.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

## Why Hyperparameter Tuning?

Every sklearn/XGBoost estimator ships with **default hyperparameters** that
are chosen to be *reasonable*, not *optimal*.
Defaults are designed to work on the widest variety of datasets possible,
so they are almost always sub-optimal for any specific problem.

**What tuning does:**

1. Defines a *search space* – a grid of plausible values for each parameter.
2. Trains and cross-validates many different combinations from that space.
3. Returns the combination that generalises best (lowest CV RMSE here).

The result is a model that is better calibrated to *this* dataset without
requiring manual trial-and-error.

In [ ]:
RF_PARAM_DIST = {
    "n_estimators":      [100, 200, 300, 500],
    "max_depth":         [None, 10, 20, 30, 40],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf":  [1, 2, 4],
    "max_features":      ["sqrt", "log2", 0.5],
    "bootstrap":         [True, False],
}

XGB_PARAM_DIST = {
    "n_estimators":     [100, 200, 300, 500],
    "max_depth":        [3, 4, 5, 6, 7, 8],
    "learning_rate":    [0.01, 0.05, 0.1, 0.2],
    "subsample":        [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.5, 0.6, 0.7, 0.8, 1.0],
    "gamma":            [0, 0.1, 0.2, 0.5],
    "reg_alpha":        [0, 0.01, 0.1, 1.0],
    "reg_lambda":       [0.5, 1.0, 2.0, 5.0],
}

### RandomForest search space

| Parameter | Values tried | Design rationale |
|---|---|---|
| `n_estimators` | 100, 200, 300, 500 | More trees → more stable predictions; diminishing returns above ~300 |
| `max_depth` | None, 10, 20, 30, 40 | `None` = fully grown trees; limiting depth is the primary knob against overfitting |
| `min_samples_split` | 2, 5, 10 | Minimum samples required to split an internal node; higher = more regularisation |
| `min_samples_leaf` | 1, 2, 4 | Minimum samples required at a leaf node; prevents ultra-fine splits on noise |
| `max_features` | sqrt, log2, 0.5 | Fraction of features considered per split; lower = more diversity between trees → lower variance |
| `bootstrap` | True, False | Whether to sample rows with replacement; `False` = use all rows per tree (lower bias, higher variance) |

### XGBoost search space

| Parameter | Values tried | Design rationale |
|---|---|---|
| `n_estimators` | 100, 200, 300, 500 | Number of boosting rounds; more = better fit but slower |
| `max_depth` | 3–8 | Depth of each tree; shallower = simpler/faster; deeper = captures more interactions |
| `learning_rate` | 0.01–0.2 | Step size shrinkage; smaller = more robust but needs more rounds |
| `subsample` | 0.6–1.0 | Row subsampling per tree; <1.0 prevents overfitting and speeds up training |
| `colsample_bytree` | 0.5–1.0 | Column subsampling per tree; <1.0 introduces diversity (similar to RF `max_features`) |
| `gamma` | 0–0.5 | Minimum loss reduction required to make a split; higher = more conservative tree |
| `reg_alpha` | 0–1.0 | L1 regularisation on leaf weights; drives sparse solutions |
| `reg_lambda` | 0.5–5.0 | L2 regularisation on leaf weights; prevents very large weights |

## Why `RandomizedSearchCV` over `GridSearchCV`?

**GridSearchCV** exhaustively evaluates *every* combination in the grid.
For RandomForest alone:

```
n_estimators × max_depth × min_samples_split × min_samples_leaf × max_features × bootstrap
= 4 × 5 × 3 × 3 × 3 × 2  =  1 080 combinations
× 5-fold CV               =  5 400 fits
```

**RandomizedSearchCV** draws `n_iter=30` *random* combinations instead:

```
30 combinations × 5-fold CV = 150 fits  (~36× faster for RF alone)
```

**Why random is good enough:**

- Bergstra & Bengio (2012) show that random search finds near-optimal
  hyperparameters in far less time than grid search, because most parameters
  have low sensitivity – the search space is effectively low-dimensional.
- Running more iterations only increases compute cost linearly, making it
  easy to balance speed vs. thoroughness with a single `n_iter` knob.

## Why 5-Fold Cross-Validation?

If we evaluated every candidate on the same held-out split we would
inadvertently *overfit to the validation set* – the winner would be the
model that got lucky on that particular split, not the one that generalises.

**5-fold CV avoids this:**

1. The training set is divided into 5 equal folds.
2. Each candidate is trained on 4 folds (80 %) and validated on the
   remaining fold (20 %).
3. This is repeated 5 times, rotating which fold is held out.
4. The **final CV score** is the mean of the 5 validation scores → a much
   more reliable estimate of how the model will perform on unseen data.

Choosing **k = 5** (rather than, say, k = 10) is a practical compromise:
it gives a low-variance score estimate while keeping compute cost manageable
(150 fits instead of 300).

In [ ]:
def _metrics(model, X_test, y_test) -> dict:
    """Compute RMSE, MAE and R² for a fitted model on the test set."""
    y_pred = model.predict(X_test)
    return {
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MAE":  mean_absolute_error(y_test, y_pred),
        "R2":   r2_score(y_test, y_pred),
    }

In [ ]:
def tune_model(
    name, estimator, param_dist,
    X_train, y_train,
    n_iter=30, cv=5, random_state=42
):
    """
    Run RandomizedSearchCV for one estimator.

    Returns the fitted SearchCV object whose `.best_estimator_` attribute
    is already retrained on the full training set (because refit=True).
    """
    print(f"\n  [{name}] Starting RandomizedSearchCV (n_iter={n_iter}, cv={cv}-fold)...")
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        refit=True,
        n_jobs=-1,
        random_state=random_state,
        verbose=1,
    )
    search.fit(X_train, y_train)
    best_cv_rmse = -search.best_score_
    print(f"  [{name}] Best CV RMSE : {best_cv_rmse:,.0f}")
    print(f"  [{name}] Best params  : {search.best_params_}")
    return search

### `refit=True` – why we don't need a manual retraining step

When `refit=True` (the default), sklearn automatically:

1. Finds the parameter combination with the best mean CV score.
2. **Retrains the estimator with those parameters on the full `X_train`**
   (all folds combined) before returning.

So `search.best_estimator_` is already a model trained on 100 % of the
training data – there is no need to call `.fit()` again.
This is important because training on more data generally improves
generalisation, and it avoids an error-prone manual step.

In [ ]:
def run_tuning(
    baseline_results, fitted_models,
    X_train, X_test, y_train, y_test,
    top_n=2, n_iter=30
):
    """
    Tune the top_n models by R² from baseline_results.

    Parameters
    ----------
    baseline_results : pd.DataFrame
        Index = model name, columns include RMSE / MAE / R2.
    fitted_models : dict
        {name: fitted_estimator} from the baseline training step.
    X_train, X_test, y_train, y_test : array-like
        The same splits used in baseline training.
    top_n : int
        Number of models to tune (ranked by R²).
    n_iter : int
        Random combinations to try per model.

    Returns
    -------
    comparison_df : pd.DataFrame
        Before/after metrics for every tuned model.
    tuned_models : dict
        {name: best_estimator_}
    """
    top_models = baseline_results.nlargest(top_n, "R2").index.tolist()
    print(f"\nTop {top_n} models selected for tuning: {top_models}")

    tunable = {
        "RandomForest": (RandomForestRegressor(random_state=42), RF_PARAM_DIST),
        "XGBoost":      (XGBRegressor(random_state=42, verbosity=0), XGB_PARAM_DIST),
    }

    records      = []
    tuned_models = {}

    for name in top_models:
        if name not in tunable:
            print(f"  Skipping '{name}' -- no search space defined.")
            continue

        estimator, param_dist = tunable[name]
        before = {
            "RMSE": baseline_results.loc[name, "RMSE"],
            "MAE":  baseline_results.loc[name, "MAE"],
            "R2":   baseline_results.loc[name, "R2"],
        }

        search     = tune_model(name=name, estimator=estimator,
                                param_dist=param_dist,
                                X_train=X_train, y_train=y_train,
                                n_iter=n_iter)
        best_model = search.best_estimator_
        tuned_models[name] = best_model
        after = _metrics(best_model, X_test, y_test)

        records.append({
            "Model":       name,
            "RMSE_before": before["RMSE"],
            "RMSE_after":  after["RMSE"],
            "RMSE_delta":  before["RMSE"] - after["RMSE"],
            "MAE_before":  before["MAE"],
            "MAE_after":   after["MAE"],
            "R2_before":   before["R2"],
            "R2_after":    after["R2"],
            "R2_delta":    after["R2"] - before["R2"],
        })

        print(f"\n  [{name}] Comparison")
        print(f"    RMSE : {before['RMSE']:>12,.0f}  ->  {after['RMSE']:>12,.0f}  "
              f"(delta {before['RMSE'] - after['RMSE']:+,.0f})")
        print(f"    MAE  : {before['MAE']:>12,.0f}  ->  {after['MAE']:>12,.0f}  "
              f"(delta {before['MAE'] - after['MAE']:+,.0f})")
        print(f"    R2   : {before['R2']:>12.4f}  ->  {after['R2']:>12.4f}  "
              f"(delta {after['R2'] - before['R2']:+.4f})")

    comparison_df = pd.DataFrame(records).set_index("Model")
    return comparison_df, tuned_models

## How to Interpret the Results

After `run_tuning()` returns, you will have a `comparison_df` with columns:

| Column | Meaning |
|---|---|
| `RMSE_before` / `RMSE_after` | Test-set RMSE before and after tuning |
| `RMSE_delta` | `before − after`; **positive = improvement** |
| `MAE_before` / `MAE_after` | Test-set MAE before and after tuning |
| `R2_before` / `R2_after` | Coefficient of determination before and after |
| `R2_delta` | `after − before`; **positive = improvement** |

**What to look for:**

- A large positive `RMSE_delta` means tuning substantially reduced the
  average prediction error in the original unit (e.g. dollars for price).
- If `RMSE_delta ≈ 0` the defaults were already near-optimal for this
  dataset, or `n_iter` was too low to explore the space adequately.
- Always compare `best_cv_rmse` (printed during search) with `RMSE_after`.
  If they are close, the model generalises well; a large gap suggests
  residual overfitting or distribution shift between train and test.

> **Tip:** pass the `tuned_models` dict directly to the evaluation notebook
> (`04_evaluation.ipynb`) to run final diagnostics on the winning model.